# How the datasets were found (harvest pipeline)

**How the journal datasets were found.** Europe PMC was searched for every open-access paper with supplementary files whose title/abstract mentions Toxoplasma, toxoplasmosis, gondii, TORCH, multiplex serology or pathogen burden (3,482 papers). Their data files (xlsx, xls, csv, sav, dta, ...) were downloaded from the PMC Open Access S3 bucket (1,407 papers had data files, 6.6 GB) and every table with >= 80 rows was scanned for a Toxoplasma result column (607 tables, 285 papers). Human, participant-level tables with at least two IgG classes were then checked by hand. Scripts: `datasets/_harvest/`.

**This notebook is not executed**: it downloads about 7 GB and scans ~1,400 papers (about 1.5 h). Run the sections in order. Its outputs (`candidates*.csv`, `shortlist.csv`, `*_hits.json`, `repo_records.json`) are already in this folder; the large caches (`supp/`, `repo_supp/`, `tables/`) are git-ignored.

Note: the first download attempt used the Europe PMC supplementaryFiles service, which was throttled to ~2 KB/s; the PMC Open Access S3 bucket (section 6) replaced it.

In [ ]:
import os, sys
while not os.path.isdir('datasets') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
os.chdir('datasets/_harvest'); print(os.getcwd())

## 1. Search Figshare, Dryad, Harvard Dataverse, Mendeley Data and OSF

Search Zenodo, Figshare, Dryad, Harvard Dataverse, Mendeley Data and OSF for Toxoplasma datasets -> repo_hits.json

_(was `repos_search.py`)_

In [ ]:
import json, re, requests

S = requests.Session(); S.headers['User-Agent'] = 'toxo-research/1.0 (academic)'
TERMS = ['toxoplasma', 'toxoplasmosis', 'toxoplasma gondii seroprevalence', 'toxoplasma IgG']
out = {}

**`zenodo`**

In [ ]:
def zenodo():
    r = {}
    for t in TERMS:
        for page in range(1, 11):
            j = S.get('https://zenodo.org/api/records', params={'q': t, 'size': 100, 'page': page}, timeout=60).json()
            hits = j['hits']['hits']
            for h in hits:
                m = h['metadata']
                r[h['id']] = dict(id=h['id'], doi=h.get('doi'), title=m['title'], type=m.get('resource_type', {}).get('type'),
                                  files=[(f['key'], f['size'], f['links']['self']) for f in h.get('files', [])])
            if len(hits) < 100:
                break
    return list(r.values())

**`figshare`**

In [ ]:
def figshare():
    r = {}
    for t in TERMS:
        for page in range(1, 11):
            j = S.post('https://api.figshare.com/v2/articles/search', json={'search_for': t, 'page': page, 'page_size': 100}, timeout=60).json()
            for a in j:
                r[a['id']] = dict(id=a['id'], doi=a.get('doi'), title=a['title'], type=a.get('defined_type_name'), url=a['url_public_html'])
            if len(j) < 100:
                break
    return list(r.values())

**`dryad`**

In [ ]:
def dryad():
    r = {}
    for t in TERMS:
        for page in range(1, 6):
            j = S.get('https://datadryad.org/api/v2/search', params={'q': t, 'per_page': 100, 'page': page}, timeout=60).json()
            d = j.get('_embedded', {}).get('stash:datasets', [])
            for x in d:
                r[x['identifier']] = dict(id=x['identifier'], title=x['title'])
            if len(d) < 100:
                break
    return list(r.values())

**`dataverse`**

In [ ]:
def dataverse():
    r = {}
    for t in TERMS:
        for start in range(0, 1000, 100):
            j = S.get('https://dataverse.harvard.edu/api/search', params={'q': t, 'type': 'dataset', 'per_page': 100, 'start': start}, timeout=60).json()
            items = j['data']['items']
            for i in items:
                r[i.get('global_id')] = dict(id=i.get('global_id'), title=i['name'])
            if len(items) < 100:
                break
    return list(r.values())

**`mendeley`**

In [ ]:
def mendeley():
    r = {}
    for t in TERMS:
        for url, p in [('https://data.mendeley.com/api/research-data/search', {'search': t, 'type': 'DATASET', 'size': 100}),
                       ('https://data.mendeley.com/public-api/datasets', {'search': t})]:
            try:
                j = S.get(url, params=p, timeout=60).json()
            except Exception:
                continue
            items = j.get('results') if isinstance(j, dict) else j
            items = items or (j.get('data', {}).get('results') if isinstance(j, dict) else [])
            for i in items or []:
                ttl = i.get('name') or i.get('title') or ''
                if re.search('toxo', ttl, re.I):
                    r[i.get('id')] = dict(id=i.get('id'), title=ttl)
    return list(r.values())

**`osf`**

In [ ]:
def osf():
    r = {}
    j = S.get('https://share.osf.io/api/v2/search/creativeworks/_search', params={'q': 'toxoplasma', 'size': 100}, timeout=60)
    try:
        for h in j.json()['hits']['hits']:
            s = h['_source']; r[h['_id']] = dict(id=h['_id'], title=s.get('title'), types=s.get('types'), url=(s.get('identifiers') or [''])[0])
    except Exception as e:
        r['err'] = dict(id='err', title=f'{j.status_code} {str(e)[:80]}')
    return list(r.values())

In [ ]:
for n, f in [('zenodo', zenodo), ('figshare', figshare), ('dryad', dryad), ('dataverse', dataverse), ('mendeley', mendeley), ('osf', osf)]:
    try:
        out[n] = f()
    except Exception as e:
        out[n] = [dict(id='err', title=str(e)[:200])]
    print(n, len(out[n]))
json.dump(out, open('repo_hits.json', 'w'), indent=1)

## 2. Search Zenodo (25 records per page)

Zenodo: all records matching Toxoplasma terms (25 per page, anonymous limit) -> zenodo_hits.json

_(was `zenodo_search.py`)_

In [ ]:
import json, requests, time

S = requests.Session(); r = {}
for t in ['toxoplasma', 'toxoplasmosis', 'gondii', 'TORCH seroprevalence']:
    for page in range(1, 60):
        j = S.get('https://zenodo.org/api/records', params={'q': t, 'size': 25, 'page': page}, timeout=60)
        if j.status_code != 200:
            time.sleep(3); continue
        hits = j.json()['hits']['hits']
        for h in hits:
            m = h['metadata']
            r[h['id']] = dict(id=h['id'], doi=h.get('doi'), title=m['title'], type=m.get('resource_type', {}).get('type'),
                              files=[(f['key'], f['size'], f['links']['self']) for f in h.get('files', [])])
        if len(hits) < 25:
            break
json.dump(list(r.values()), open('zenodo_hits.json', 'w'), indent=1)
print('zenodo records', len(r))

## 3. Search Mendeley Data

Mendeley Data search (query=, 10 per page) for Toxoplasma terms -> mendeley_hits.json

_(was `mendeley_search.py`)_

In [ ]:
import json, requests

S = requests.Session(); out = {}
for term in ['toxoplasma', 'toxoplasmosis', 'gondii']:
    for page in range(1, 80):
        r = S.get('https://data.mendeley.com/api/research-data/search', params={'query': term, 'size': 10, 'page': page}, timeout=60)
        if r.status_code != 200:
            break
        j = r.json()
        recs = j.get('records', [])
        for r in recs:
            out[r['id']] = dict(id=r['id'], title=r['title'], doi=(r.get('doi') or {}).get('id') if isinstance(r.get('doi'), dict) else r.get('doi'),
                                version=r.get('version'), url=r.get('url'), source=r.get('source'), type=r.get('type'))
        if len(recs) < 10:
            break
json.dump(list(out.values()), open('mendeley_hits.json', 'w'), indent=1)
print(len(out))
for r in out.values():
    if 'oxo' in r['title'] or 'gondii' in r['title']:
        print(r['id'], r.get('source'), r.get('version'), '|', r['title'][:120])

Download the human Mendeley datasets found above

In [ ]:
import requests, json, re
S = requests.Session(); recs = []
for i in ['m4hvf5g7mj', 'dxg3498c7y', 'mcw665mzrp', 's84shxhhfv', '9smgkth8r7']:
    j = S.get(f'https://data.mendeley.com/public-api/datasets/{i}', timeout=60).json()
    d = f'repo_supp/mendeley_data/{i}'; os.makedirs(d, exist_ok=True)
    for f in j.get('files', []):
        if re.search(r'\.(xlsx|xls|csv|sav|dta|txt|ods|zip)$', f['filename'], re.I):
            open(os.path.join(d, f['filename']), 'wb').write(S.get(f['content_details']['download_url'], timeout=120).content)
    recs.append(dict(source='mendeley_data', id=i, title=j['name'], doi=j['doi']['id'], files=[f['filename'] for f in j.get('files', [])]))
json.dump(recs, open('mendeley_records.json', 'w'), indent=1)

## 4. Europe PMC search 1: open-access Toxoplasma serology papers with supplements

Europe PMC: every open-access Toxoplasma serology paper with supplementary files -> hits_all.json, hits_human.json

_(was `epmc_search.py`)_

In [ ]:
import json, re, requests
Q = ('(TITLE_ABS:"toxoplasma" OR TITLE_ABS:"toxoplasmosis") AND (seroprevalence OR seropositivity OR "risk factors" '
     'OR "IgG" OR serological OR antibodies) AND HAS_SUPPL:Y AND OPEN_ACCESS:Y')
hits, cur = [], '*'
while True:
    j = requests.get('https://www.ebi.ac.uk/europepmc/webservices/rest/search',
                     params={'query': Q, 'format': 'json', 'pageSize': 1000, 'cursorMark': cur, 'resultType': 'lite'}, timeout=120).json()
    hits += j['resultList']['result']
    if j.get('nextCursorMark') in (None, cur) or not j['resultList']['result']:
        break
    cur = j['nextCursorMark']
json.dump(hits, open('hits_all.json', 'w'))
ANIMAL = re.compile(r'\b(cats?|dogs?|sheep|goats?|cattle|pigs?|swine|chickens?|poultry|horses?|donkeys?|camels?|deer|boars?|rodents?|birds?|wildlife|wild|animals?|livestock|bovine|ovine|caprine|feline|canine|rabbits?|rats?|mice|mouse|marine|seals?|otters?|dolphins?|bats?|monkeys?|primates?|ruminants?|equids?|buffalo|yaks?|alpacas?|llamas?|ferrets?|foxes|fox|lynx|wolves|wolf|bears?|kangaroos?|marsupials?|antelopes?|mink|ducks?|pigeons?|turkeys?|ostrich|fish|meat|pork|mutton|game|hares?|zoo|reindeer|moose|water buffalo|beluga|whales?|cetaceans?|squirrels?|hedgehogs?)\b', re.I)
HUMAN = re.compile(r'\b(humans?|people|women|woman|pregnan\w*|patients?|donors?|students?|child\w*|individuals?|adults?|residents?|population|workers?|men|participants?|hiv|psychiatric|schizophren\w*|community|households?|veterinarians?|butchers?|abattoir|slaughterhouse|farmers?|inmates?|prisoners?|soldiers?|refugees?|migrants?|mothers?|newborns?|infants?|hunters?|cohort|epilepsy|cancer|transplant\w*|haemodialysis|hemodialysis|diabet\w*|suicid\w*|depress\w*|bipolar|autism|alzheimer|parkinson|ocular|uveitis|retinochoroiditis|nation\w*|general)\b', re.I)
SERO = re.compile(r'seroprev|seropos|serolog|antibod|igg|risk factor|prevalence|infection|exposure|associated|association|survey|cross-sectional|case-control|immunity|seroepidem', re.I)
human = [h for h in hits if SERO.search(h.get('title', '')) and (HUMAN.search(h.get('title', '')) or not ANIMAL.search(h.get('title', '')))]
human = [h for h in human if h.get('pmcid')]
json.dump(human, open('hits_human.json', 'w'))
print('all', len(hits), '| human serology w/ PMCID', len(human))

## 5. Europe PMC search 2: add TORCH / multiplex serology / pathogen burden papers (no title filter)

Broader Europe PMC search: Toxoplasma OR TORCH OR multiplex/pathogen-burden serology, open access with supplements.
Merges with the first search -> hits_scan.json (no title filter: the table scanner decides).

_(was `epmc_search2.py`)_

In [ ]:
import json, requests
Q = ('(TITLE_ABS:"toxoplasma" OR TITLE_ABS:"toxoplasmosis" OR TITLE_ABS:"gondii" OR TITLE_ABS:"TORCH" OR '
     'TITLE_ABS:"multiplex serology" OR TITLE_ABS:"pathogen burden" OR TITLE_ABS:"infectious burden" OR '
     '(TITLE_ABS:"seroprevalence" AND (TITLE_ABS:"rubella" OR TITLE_ABS:"cytomegalovirus" OR TITLE_ABS:"parasitic"))) '
     'AND HAS_SUPPL:Y AND OPEN_ACCESS:Y')
hits, cur = [], '*'
while True:
    j = requests.get('https://www.ebi.ac.uk/europepmc/webservices/rest/search',
                     params={'query': Q, 'format': 'json', 'pageSize': 1000, 'cursorMark': cur, 'resultType': 'lite'}, timeout=180).json()
    hits += j['resultList']['result']
    if j.get('nextCursorMark') in (None, cur) or not j['resultList']['result']:
        break
    cur = j['nextCursorMark']
old = json.load(open('hits_all.json'))
allh = {h['pmcid']: h for h in old + hits if h.get('pmcid')}
json.dump(list(allh.values()), open('hits_scan.json', 'w'))
print('broad query', len(hits), '| merged with PMCID', len(allh))

## 6. Download the data files of every paper from the PMC Open Access S3 bucket

Download data-type supplementary files (xlsx/xls/csv/sav/dta/...) for every candidate paper from the PMC Open Access
S3 bucket (https://pmc-oa-opendata.s3.amazonaws.com) -> supp/<PMCID>/<file>.
Usage (from datasets/_harvest):  python pmc_s3_download.py hits_human.json

_(was `pmc_s3_download.py`)_

In [ ]:
import json, os, re, sys, concurrent.futures as cf, requests
B = 'https://pmc-oa-opendata.s3.amazonaws.com'
DATA = re.compile(r'\.(xlsx|xls|xlsm|csv|tsv|txt|sav|dta|ods|zip|rds|rdata|sas7bdat|xpt|por)$', re.I)
S = requests.Session()
hits = json.load(open(sys.argv[1] if len(sys.argv) > 1 else 'hits_human.json'))

**`keys`**

In [ ]:
def keys(prefix):
    r = S.get(B, params={'list-type': 2, 'prefix': prefix, 'max-keys': 1000}, timeout=60).text
    return list(zip(re.findall(r'<Key>([^<]+)</Key>', r), map(int, re.findall(r'<Size>(\d+)</Size>', r))))

**`get`**

In [ ]:
def get(pmc):
    d = f'supp/{pmc}'
    if os.path.exists(d + '/.done'):
        return
    os.makedirs(d, exist_ok=True)
    try:
        ks = keys(pmc + '.')
        vers = sorted({k.split('/')[0] for k, _ in ks}, key=lambda v: int(v.split('.')[-1]))
        if not vers:
            open(d + '/.done', 'w').write('not in OA bucket'); return
        for k, size in ks:
            name = os.path.basename(k)
            if k.startswith(vers[-1] + '/') and DATA.search(name) and not name.startswith(vers[-1]) and size < 100_000_000:
                p = os.path.join(d, name)
                if not os.path.exists(p):
                    open(p, 'wb').write(S.get(f'{B}/{k}', timeout=300).content)
        open(d + '/.done', 'w').write('ok')
    except Exception as e:
        print(pmc, e)

In [ ]:
if True:   # in the notebook: run the download
    with cf.ThreadPoolExecutor(16) as ex:
        list(ex.map(get, [h['pmcid'] for h in hits]))
    n = [p for p in os.listdir('supp') if any(not f.startswith('.') for f in os.listdir('supp/' + p))]
    print('papers with data files:', len(n), 'of', len(hits))

## 7. Download data files of human Toxoplasma records on Figshare, Zenodo, Dataverse and Dryad

Download data files from repository records that look like human Toxoplasma studies.
Figshare (API file lists), Zenodo (record files), Harvard Dataverse (dataset files), Dryad (via its Zenodo mirror)
-> repo_supp/<source>/<record id>/<file>, plus repo_records.json (title, doi, source).

_(was `repo_download.py`)_

In [ ]:
import json, os, re, concurrent.futures as cf, requests

S = requests.Session(); S.headers['User-Agent'] = 'toxo-research/1.0 (academic)'
DATA = re.compile(r'\.(xlsx|xls|xlsm|csv|tsv|txt|sav|dta|ods|zip|rds|rdata|sas7bdat|por)$', re.I)
TOX = re.compile(r'toxo|gondii|torch', re.I)
NOT_HUMAN = re.compile(r'\b(cats?|dogs?|sheep|goats?|cattle|pigs?|swine|chickens?|poultry|horses?|deer|cervids?|wild|wildlife|rodents?|'
                       r'mice|mouse|murine|rats?|birds?|livestock|animals?|boars?|camels?|zoo|otters?|hyenas?|chimpanzees?|gulls?|'
                       r'caribou|lynx|bats?|proteom\w*|transcriptom\w*|genom\w*|crispr|kinase|protein|strains?|isolates?|'
                       r'tachyzoites?|bradyzoites?|organelle|apicoplast|rhoptry|microscopy|mzml|vaccine|epitope|in silico|'
                       r'drug|inhibitor|compound|cells?|oocysts?|salad|meat|soil|water samples|genotyp\w*|sequenc\w*|'
                       r'figure|fig\.|meta-analysis|systematic review|torch-|pytorch|\btorch\b(?!.*(infect|pregnan|seroprev)))\b', re.I)
recs = []

**`keep`**

In [ ]:
def keep(title):
    return bool(TOX.search(title or '')) and not NOT_HUMAN.search(title or '')

**`save`**

In [ ]:
def save(source, rid, title, doi, files):
    d = f'repo_supp/{source}/{rid}'
    os.makedirs(d, exist_ok=True)
    n = 0
    for name, url, size in files:
        if DATA.search(name) and (size or 0) < 80_000_000:
            p = os.path.join(d, re.sub(r'[/\\]', '_', name))
            if not os.path.exists(p):
                r = S.get(url, timeout=300)
                if r.status_code == 200:
                    open(p, 'wb').write(r.content); n += 1
            else:
                n += 1
    recs.append(dict(source=source, id=str(rid), title=title, doi=doi, n_data_files=n))

**`figshare`**

In [ ]:
def figshare():
    hits = json.load(open('repo_hits.json'))['figshare']
    todo = [h for h in hits if keep(h['title'])]
    def one(h):
        try:
            a = S.get(f'https://api.figshare.com/v2/articles/{h["id"]}', timeout=60).json()
            save('figshare', h['id'], re.sub('<[^>]+>', '', h['title']), a.get('doi'),
                 [(f['name'], f['download_url'], f.get('size')) for f in a.get('files', [])])
        except Exception as e:
            print('figshare', h['id'], e)
    with cf.ThreadPoolExecutor(8) as ex:
        list(ex.map(one, todo))

**`zenodo`**

In [ ]:
def zenodo():
    hits = json.load(open('zenodo_hits.json'))
    for h in hits:
        if keep(h['title']):
            try:
                save('zenodo', h['id'], h['title'], h.get('doi'), [(f[0], f[2], f[1]) for f in h['files']])
            except Exception as e:
                print('zenodo', h['id'], e)

**`dataverse`**

In [ ]:
def dataverse():
    hits = json.load(open('repo_hits.json'))['dataverse']
    for h in hits:
        if keep(h['title']) and 'NHANES' not in h['title'] and 'National Health' not in h['title']:
            try:
                j = S.get('https://dataverse.harvard.edu/api/datasets/:persistentId/', params={'persistentId': h['id']}, timeout=60).json()
                files = j['data']['latestVersion']['files']
                save('harvard_dataverse', h['id'].split('/')[-1], h['title'], h['id'],
                     [(f['dataFile'].get('originalFileName') or f['label'],
                       f'https://dataverse.harvard.edu/api/access/datafile/{f["dataFile"]["id"]}?format=original',
                       f['dataFile'].get('filesize')) for f in files])
            except Exception as e:
                print('dataverse', h['id'], e)

**`dryad`** — Dryad human Toxoplasma datasets, downloaded from their Zenodo mirror records (same files).

In [ ]:
def dryad():
    """Dryad human Toxoplasma datasets, downloaded from their Zenodo mirror records (same files)."""
    hits = json.load(open('repo_hits.json'))['dryad']
    z = {x['doi']: x for x in json.load(open('zenodo_hits.json')) if x.get('doi')}
    for h in hits:
        doi = h['id'].replace('doi:', '')
        if keep(h['title']):
            m = z.get(doi)
            if m is None:
                q = S.get('https://zenodo.org/api/records', params={'q': f'doi:"{doi}"', 'size': 5}, timeout=60).json()['hits']['hits']
                m = q and dict(files=[(f['key'], f['size'], f['links']['self']) for f in q[0].get('files', [])])
            if m:
                save('dryad', doi.replace('/', '_'), h['title'], doi, [(f[0], f[2], f[1]) for f in m['files']])
            else:
                recs.append(dict(source='dryad', id=doi, title=h['title'], doi=doi, n_data_files=0))

In [ ]:
for f in (figshare, zenodo, dataverse, dryad):
    f(); print(f.__name__, sum(r['source'] == f.__name__.replace('dataverse', 'harvard_dataverse') for r in recs))
json.dump(recs, open('repo_records.json', 'w'), indent=1)

## 8. Scan every table for a Toxoplasma result column

Scan every downloaded supplementary data file for participant-level tables that contain a Toxoplasma result column.
-> candidates.csv (one row per table: paper, publisher, file, sheet, shape, header row, toxo-like columns + values)
   tables/<PMCID>__<file>__<sheet>.pkl (the parsed table, header applied)

_(was `scan_tables.py`)_

In [ ]:
import glob, io, json, os, re, sys, warnings, zipfile
import pandas as pd
warnings.filterwarnings('ignore')

# Run three times: ('supp/*/*', 'candidates.csv'), ('repo_supp/*/*/*', 'candidates_repos.csv'),
# ('repo_supp/mendeley_data/*/*', 'candidates_mendeley.csv')
PATTERN, OUT = 'supp/*/*', 'candidates.csv'
REPO = 'repo_supp' in PATTERN
if REPO:
    hits = {r['id']: dict(doi=r['doi'], title=r['title'], journalTitle=r['source'], pubYear='') for r in json.load(open('repo_records.json'))}
else:
    hits = {h['pmcid']: h for h in json.load(open('hits_scan.json'))}
LAB = re.compile(r'toxo|gondii|\btg\b|t\.?\s*gondii|tg[-_ ]?ig|igg|sero|result|status|elisa|infect', re.I)
os.makedirs('tables', exist_ok=True)
PUB = [('10.1371', 'plos'), ('10.1186', 'bmc_springer'), ('10.1007', 'bmc_springer'), ('10.1038', 'bmc_springer'),
       ('10.3389', 'frontiers'), ('10.3390', 'mdpi')]

**`publisher`**

In [ ]:
def publisher(doi):
    for pre, p in PUB:
        if (doi or '').startswith(pre):
            return p
    return 'other_journals'

**`header_row`**

In [ ]:
def header_row(raw):
    best, br = -1, 0
    for i in range(min(8, len(raw))):
        r = raw.iloc[i]
        score = sum(isinstance(v, str) and not re.fullmatch(r'[\d.,\s-]+', v) for v in r) - 0.5 * r.isna().sum()
        if score > best:
            best, br = score, i
    return br

**`frames_from`**

In [ ]:
def frames_from(name, data):
    ext = os.path.splitext(name)[1].lower()
    if ext == '.zip':
        z = zipfile.ZipFile(io.BytesIO(data))
        for n in z.namelist():
            if not n.endswith('/') and z.getinfo(n).file_size < 100_000_000:
                yield from frames_from(n, z.read(n))
        return
    if ext in ('.xlsx', '.xls', '.xlsm', '.ods'):
        for sh, d in pd.read_excel(io.BytesIO(data), sheet_name=None, header=None).items():
            yield name, str(sh), d
    elif ext in ('.csv', '.tsv', '.txt'):
        yield name, 'csv', pd.read_csv(io.BytesIO(data), header=None, sep=None, engine='python', on_bad_lines='skip', encoding_errors='ignore')
    elif ext in ('.sav', '.por', '.dta', '.sas7bdat', '.xpt'):
        import pyreadstat, tempfile
        with tempfile.NamedTemporaryFile(suffix=ext) as f:
            f.write(data); f.flush()
            reader = {'.sav': pyreadstat.read_sav, '.por': pyreadstat.read_por, '.dta': pyreadstat.read_dta,
                      '.sas7bdat': pyreadstat.read_sas7bdat, '.xpt': pyreadstat.read_xport}[ext]
            d, meta = reader(f.name, apply_value_formats=True)
        yield name, 'data', pd.concat([pd.DataFrame([d.columns], columns=d.columns), d], ignore_index=True)

In [ ]:
rows = []
for path in sorted(glob.glob(PATTERN)):
    parts = path.split('/')
    pmc = parts[2] if REPO else parts[1]; h = hits.get(pmc, {})
    try:
        frames = list(frames_from(os.path.basename(path), open(path, 'rb').read()))
    except Exception:
        continue
    for name, sh, raw in frames:
        if raw.shape[0] < 80 or raw.shape[1] < 3:
            continue
        hr = header_row(raw)
        d = raw.iloc[hr + 1:].copy()
        cols, seen = [], {}
        for c in raw.iloc[hr]:
            c = str(c).strip(); seen[c] = seen.get(c, 0) + 1
            cols.append(c if seen[c] == 1 else f'{c}.{seen[c]}')
        d.columns = cols
        d = d.dropna(how='all')
        if len(d) < 80:
            continue
        labs = [c for c in d.columns if LAB.search(c)]
        if not labs:
            continue
        key = f'{pmc}__{re.sub(r"[^A-Za-z0-9.]", "_", os.path.basename(name))}__{re.sub(r"[^A-Za-z0-9]", "_", sh)}'
        d.to_pickle(f'tables/{key}.pkl')
        vals = {c: d[c].astype(str).value_counts().head(4).to_dict() for c in labs[:6]}
        rows.append(dict(key=key, pmcid=pmc, publisher=parts[1] if REPO else publisher(h.get('doi')), doi=h.get('doi'), journal=h.get('journalTitle'),
                         year=h.get('pubYear'), title=h.get('title', '')[:160], file=os.path.basename(name), sheet=sh,
                         rows=len(d), cols=d.shape[1], label_candidates=json.dumps(vals, ensure_ascii=False)[:700],
                         header=' | '.join(map(str, d.columns))[:600]))
pd.DataFrame(rows).to_csv(OUT, index=False)
print(len(rows), 'candidate tables from', len({r['pmcid'] for r in rows}), 'papers')
print(pd.DataFrame(rows).groupby('publisher').pmcid.nunique())

## 9. Shortlist human tables with a binary Toxoplasma label (then checked by hand)



_(was `shortlist.py`)_

In [ ]:
import pandas as pd, re, json
c = pd.read_csv('candidates.csv')
ANI = re.compile(r'\b(cats?|dogs?|sheep|goats?|cattle|pigs?|swine|chickens?|poultry|horses?|deer|cervids?|wild|wildlife|rodents?|mice|mouse|murine|rats?|birds?|livestock|animals?|boars?|camels?|zoo|otters?|hyenas?|primates?|gulls?|caribou|lynx|bats?|felids?|ruminants?|bovine|ovine|caprine|feline|canine|marine|seals?|equids?|donkeys?|rabbits?|hares?|penguins?|ferrets?|meat|pork|game|slaughtered|abattoirs?)\b', re.I)
HUM = re.compile(r'\b(humans?|people|women|woman|pregnan\w*|patients?|donors?|students?|child\w*|individuals?|adults?|residents?|population|workers?|men|participants?|hiv|psychiatric|schizophren\w*|community|households?|veterinarians?|butchers?|farmers?|inmates?|soldiers?|refugees?|mothers?|newborns?|hunters?|cohort|persons?|homeless|cancer|epilep\w*|suicid\w*)\b', re.I)
LAB = re.compile(r'toxo|gondii|\btg\b|tg[-_ ]?ig|t\.\s*g', re.I)
GEN = re.compile(r'igg|sero|elisa|result|status|infect|antibod', re.I)
rows = []
for _, r in c.iterrows():
    t = str(r.title)
    if ANI.search(t) and not HUM.search(t):
        continue
    d = pd.read_pickle(f'tables/{r.key}.pkl')
    if len(d) > 60000:
        continue
    toxotitle = bool(re.search('toxo|gondii', t, re.I))
    for col in d.columns:
        if not (LAB.search(col) or (toxotitle and GEN.search(col))):
            continue
        if re.search(r'igm|avid|pcr|dna|index|od\b|titer|titre', col, re.I):
            continue
        v = d[col].astype(str).str.strip().str.lower().replace({'nan': None, 'none': None, '': None}).dropna()
        vc = v.value_counts()
        if 2 <= len(vc) <= 5 and vc.iloc[:2].min() >= 10 and vc.iloc[:2].sum() >= 0.85 * len(v) and len(v) >= 80:
            rows.append(dict(pmcid=r.pmcid, publisher=r.publisher, title=t[:110], key=r.key, rows=len(d), cols=d.shape[1],
                             label=col, values=json.dumps(vc.head(4).to_dict(), ensure_ascii=False)))
s = pd.DataFrame(rows).drop_duplicates(['pmcid', 'key', 'label'])
s.to_csv('shortlist.csv', index=False)
print(len(s), 'label candidates in', s.key.nunique(), 'tables /', s.pmcid.nunique(), 'papers')
for pmc, g in s.groupby('pmcid', sort=False):
    print(f"\n{pmc} [{g.publisher.iloc[0]}] {g.title.iloc[0]}")
    for _, x in g.head(4).iterrows():
        print(f"   {x.key[len(pmc)+2:][:48]} {x.rows}x{x.cols} :: {x.label[:30]} = {x['values'][:90]}")

## 10. Print a candidate file compactly



_(was `show.py`)_

In [ ]:
import pandas as pd, json, sys
c = pd.read_csv('candidates_repos.csv')   # any candidates*.csv
for pmc,g in c.groupby('pmcid',sort=False):
    r=g.iloc[0]
    print(f"\n{pmc} [{r.publisher}] {str(r.title)[:100]}")
    for _,x in g.iterrows():
        try: lc=json.loads(x.label_candidates); s=' ; '.join(f"{k[:26]}={dict(list(v.items())[:3])}" for k,v in lc.items())
        except Exception: s=str(x.label_candidates)
        print(f"   {x.key[len(str(pmc))+2:][:50]} {x.rows}x{x.cols} :: {s[:260]}")